In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "MCQ_E.xpt"
csv_file = DATA_DIR / "MCQ_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep completely untouched source
source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 MCQ_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "MCQ010",
    "MCQ025",
    "MCQ035",
    "MCQ040",
    "MCQ050",
    "MCQ051",
    "MCQ053",
    "MCQ080",
    "MCQ092",
    "MCD093",
    "MCQ140",
    "MCQ149",
    "MCQ150G",
    "MCQ150Q",
    "MCQ160A",
    "MCQ180A",
    "MCQ190",
    "MCQ160N",
    "MCQ180N",
    "MCQ160B",
    "MCQ180B",
    "MCQ160C",
    "MCQ180C",
    "MCQ160D",
    "MCQ180D",
    "MCQ160E",
    "MCQ180E",
    "MCQ160F",
    "MCQ180F",
    "MCQ160G",
    "MCQ180G",
    "MCQ160M",
    "MCQ170M",
    "MCQ180M",
    "MCQ160K",
    "MCQ170K",
    "MCQ180K",
    "MCQ160L",
    "MCQ170L",
    "MCQ180L",
    "MCQ220",
    "MCQ230A",
    "MCQ230B",
    "MCQ230C",
    "MCQ230D",
    "MCQ240A",
    "MCQ240B",
    "MCQ240C",
    "MCQ240D",
    "MCQ240E",
    "MCQ240F",
    "MCQ240G",
    "MCQ240H",
    "MCQ240I",
    "MCQ240J",
    "MCQ240K",
    "MCQ240L",
    "MCQ240M",
    "MCQ240N",
    "MCQ240O",
    "MCQ240P",
    "MCQ240Q",
    "MCQ240R",
    "MCQ240S",
    "MCQ240T",
    "MCQ240U",
    "MCQ240V",
    "MCQ240W",
    "MCQ240X",
    "MCQ240Y",
    "MCQ240Z",
    "MCQ240AA",
    "MCQ240BB",
    "MCQ240CC",
    "MCQ240DD",
    "MCQ240DK",
    "MCQ245A",
    "MCQ245B",
    "MCQ300A",
    "MCQ300B",
    "MCQ300C",
    "MCQ265",
    "MCQ268A",
    "MCQ268B",
    "MCQ268C",
    "MCQ268D",
    "MCQ310",
    "MCQ320",
    "MCD330",
    "MCQ340",
    "MCQ350"
]


if df.shape != (9666, 92):

    raise ValueError(
        f"Unexpected MCQ_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "MCQ_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():

    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: MCQ_E structure = "
    "9,666 rows x 92 columns."
)


# ============================================================
# 4. NUMERIC FIELD CHECK
# ============================================================

#
# Actual MCQ_E XPT contains only numeric SAS variables.
#

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


non_numeric_cols = [
    col
    for col in df.columns
    if col not in numeric_cols
]


if non_numeric_cols:

    raise ValueError(
        "Unexpected nonnumeric fields found:\n"
        + str(non_numeric_cols)
    )


print(
    "PASS: All 92 released variables are numeric."
)


# ============================================================
# 5. KNOWN SAS/XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)

print(
    "Tiny artifacts:",
    f"{tiny_total:,}"
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY PATTERN
# ============================================================

EXPECTED_TINY_COUNTS = {
    "MCQ150Q": 721,
    "MCQ245B": 2124,
    "MCQ340": 39
}


EXPECTED_TINY_TOTAL = 2884


actual_tiny_counts = {
    col: int(count)
    for col, count in tiny_counts.items()
    if count > 0
}


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny artifact pattern differs.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny-artifact count."
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected true zeros already decoded "
        "in the original XPT."
    )


print(
    "\nPASS: Exact MCQ_E tiny pattern confirmed."
)

print(
    "      MCQ150Q = 721"
)

print(
    "      MCQ245B = 2,124"
)

print(
    "      MCQ340  = 39"
)

print(
    "      TOTAL   = 2,884"
)


# ============================================================
# 7. CDC ZERO-SAFETY CHECK
# ============================================================

#
# CDC:
#
# MCQ150Q
# # school days missed
# "IF NONE, ENTER 0"
# Valid released range begins at 0.
#
#
# MCQ245B
# # workdays missed
# Valid released range = 0-365.
#
#
# MCQ340
# # PSA tests in past 5 years
# Valid released range = 0-20.
#
#
# ONLY these three columns are approved for
# tiny -> zero restoration.
#

APPROVED_ZERO_COLUMNS = {
    "MCQ150Q",
    "MCQ245B",
    "MCQ340"
}


unexpected_tiny_columns = (
    set(actual_tiny_counts)
    -
    APPROVED_ZERO_COLUMNS
)


if unexpected_tiny_columns:

    raise ValueError(
        "STOP: Tiny values found in unexpected variables:\n"
        + str(
            sorted(unexpected_tiny_columns)
        )
    )


print(
    "PASS: All tiny values occur only in "
    "CDC zero-permitted fields."
)


# ============================================================
# 8. RESTORE VALIDATED ZEROS
# ============================================================

print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


for col, expected_count in (
    EXPECTED_TINY_COUNTS.items()
):

    mask = df[col].eq(
        TINY_VALUE
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} "
            f"tiny values, found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<10}: "
        f"{actual_count:,} tiny -> 0"
    )


print(
    "PASS: All 2,884 validated zeros restored."
)


# ============================================================
# 9. VERIFY EXACT ZERO COUNTS
# ============================================================

EXPECTED_ZERO_COUNTS = {
    "MCQ150Q": 721,
    "MCQ245B": 2124,
    "MCQ340": 39
}


for col, expected_count in (
    EXPECTED_ZERO_COUNTS.items()
):

    actual_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: zero count differs.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


total_final_zeros = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


if total_final_zeros != 2884:

    raise ValueError(
        "Unexpected zeros exist outside the "
        "three approved MCQ variables."
    )


print(
    "PASS: Final zero counts are exact."
)

print(
    "PASS: No unexpected zeros introduced."
)


# ============================================================
# 10. VERIFY NO TINY VALUE REMAINS
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 11. SEQN VALIDATION
# ============================================================

seqn = df["SEQN"]


if int(
    seqn.isna().sum()
) != 0:

    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated().sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN."
    )


if seqn.min() != 41475:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "\nPASS: SEQN range = 41475-51623."
)

print(
    "PASS: Missing SEQN = 0."
)

print(
    "PASS: Duplicate SEQN = 0."
)


# ============================================================
# 12. FRACTIONAL / DECIMAL AUDIT
# ============================================================

#
# IMPORTANT:
#
# After the three validated tiny -> zero repairs,
# the actual attached MCQ_E has NO genuinely
# fractional variables.
#
# Therefore:
#
# genuine decimal variables = 0
#
# This is specific to MCQ_E.
#
# It does NOT change the broader rule that genuine
# dietary/lab/etc. decimals are always preserved AS IS.
#

fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:

        fractional_columns[
            col
        ] = fractional_count


print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)

print(
    "Genuine fractional variables:",
    fractional_columns
)


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal values found:\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: Genuine decimal variables = 0."
)


# ============================================================
# 13. CDC CHECK - MCQ150Q
# ============================================================

#
# CDC:
#
# 0-100   = 2,364
# 77777   = 0
# 99999   = 3
# Missing = 7,299
#
# Attached XPT zero count after correction = 721.
#

s = df["MCQ150Q"]


if int(
    s.between(
        0,
        100
    )
    .sum()
) != 2364:

    raise ValueError(
        "MCQ150Q 0-100 count differs."
    )


if int(
    s.eq(0)
    .sum()
) != 721:

    raise ValueError(
        "MCQ150Q zero count differs."
    )


if int(
    s.eq(77777)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected MCQ150Q=77777."
    )


if int(
    s.eq(99999)
    .sum()
) != 3:

    raise ValueError(
        "MCQ150Q=99999 count differs."
    )


if int(
    s.isna()
    .sum()
) != 7299:

    raise ValueError(
        "MCQ150Q missing count differs."
    )


print(
    "\nPASS: MCQ150Q"
)

print(
    "      0-100 = 2,364"
)

print(
    "      of which zero = 721"
)

print(
    "      99999 = 3"
)

print(
    "      Missing = 7,299"
)


# ============================================================
# 14. CDC CHECK - MCQ245B
# ============================================================

#
# CDC:
#
# 0-365   = 3,899
# 77777   = 0
# 99999   = 1
# Missing = 5,766
#
# Attached XPT zero count after correction = 2,124.
#

s = df["MCQ245B"]


if int(
    s.between(
        0,
        365
    )
    .sum()
) != 3899:

    raise ValueError(
        "MCQ245B 0-365 count differs."
    )


if int(
    s.eq(0)
    .sum()
) != 2124:

    raise ValueError(
        "MCQ245B zero count differs."
    )


if int(
    s.eq(77777)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected MCQ245B=77777."
    )


if int(
    s.eq(99999)
    .sum()
) != 1:

    raise ValueError(
        "MCQ245B=99999 count differs."
    )


if int(
    s.isna()
    .sum()
) != 5766:

    raise ValueError(
        "MCQ245B missing count differs."
    )


print(
    "\nPASS: MCQ245B"
)

print(
    "      0-365 = 3,899"
)

print(
    "      of which zero = 2,124"
)

print(
    "      99999 = 1"
)

print(
    "      Missing = 5,766"
)


# ============================================================
# 15. CDC CHECK - MCQ340
# ============================================================

#
# CDC:
#
# 0-20    = 1,042
# 777     = 0
# 999     = 47
# Missing = 8,577
#
# Attached XPT zero count after correction = 39.
#

s = df["MCQ340"]


if int(
    s.between(
        0,
        20
    )
    .sum()
) != 1042:

    raise ValueError(
        "MCQ340 0-20 count differs."
    )


if int(
    s.eq(0)
    .sum()
) != 39:

    raise ValueError(
        "MCQ340 zero count differs."
    )


if int(
    s.eq(777)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected MCQ340=777."
    )


if int(
    s.eq(999)
    .sum()
) != 47:

    raise ValueError(
        "MCQ340=999 count differs."
    )


if int(
    s.isna()
    .sum()
) != 8577:

    raise ValueError(
        "MCQ340 missing count differs."
    )


print(
    "\nPASS: MCQ340"
)

print(
    "      0-20 = 1,042"
)

print(
    "      of which zero = 39"
)

print(
    "      999 = 47"
)

print(
    "      Missing = 8,577"
)


# ============================================================
# 16. CDC ANCHOR CHECK - MCQ010
# ============================================================

EXPECTED_MCQ010 = {
    1: 1385,
    2: 8274,
    7: 1,
    9: 6
}


actual_mcq010 = (
    df["MCQ010"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual_mcq010 = {
    int(k): int(v)
    for k, v in actual_mcq010.items()
}


if actual_mcq010 != EXPECTED_MCQ010:

    raise ValueError(
        "MCQ010 frequency mismatch.\n"
        f"Expected: {EXPECTED_MCQ010}\n"
        f"Actual:   {actual_mcq010}"
    )


if int(
    df["MCQ010"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "MCQ010 missing count differs."
    )


print(
    "\nPASS: MCQ010 matches CDC."
)


# ============================================================
# 17. CDC ANCHOR CHECK - MCQ220
# ============================================================

#
# Ever told had cancer/malignancy
#
# CDC:
#
# Yes        = 576
# No         = 5,348
# Refused    = 0
# Don't know = 11
# Missing    = 3,731
#

EXPECTED_MCQ220 = {
    1: 576,
    2: 5348,
    9: 11
}


actual_mcq220 = (
    df["MCQ220"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual_mcq220 = {
    int(k): int(v)
    for k, v in actual_mcq220.items()
}


if actual_mcq220 != EXPECTED_MCQ220:

    raise ValueError(
        "MCQ220 frequency mismatch.\n"
        f"Expected: {EXPECTED_MCQ220}\n"
        f"Actual:   {actual_mcq220}"
    )


if int(
    df["MCQ220"]
    .eq(7)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected MCQ220=7."
    )


if int(
    df["MCQ220"]
    .isna()
    .sum()
) != 3731:

    raise ValueError(
        "MCQ220 missing count differs."
    )


print(
    "PASS: MCQ220 matches CDC."
)


# ============================================================
# 18. BUILD EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# expected_df =
#
# ORIGINAL RELEASED XPT
#
# +
# ONLY:
#
# MCQ150Q tiny -> 0
# MCQ245B tiny -> 0
# MCQ340  tiny -> 0
#
# NOTHING ELSE.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 19. VERIFY EVERY RELEASED VARIABLE IS WHOLE
# ============================================================

#
# This must happen BEFORE converting to Int64.
#

for col in EXPECTED_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    if not np.all(
        np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=0
        )
    ):

        raise ValueError(
            f"STOP: {col} contains a non-whole value."
        )


print(
    "\nPASS: All 92 MCQ_E variables contain "
    "whole-number source values after correction."
)


# ============================================================
# 20. CLEAN WHOLE-NUMBER FORMATTING
# ============================================================

#
# Since all variables have now been proven whole:
#
# SEQN
# 41475.0 -> 41475
#
# MCQ010
# 1.0 -> 1
#
# MCQ150Q
# tiny -> 0
#
# MCQ245B
# tiny -> 0
#
# MCQ340
# tiny -> 0
#
# MCD330
# 365.0 -> 365
#
# Missing -> blank
#

for col in EXPECTED_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: All MCQ_E variables converted "
    "to clean integer-style formatting."
)


# ============================================================
# 21. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    equal = np.isclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not bool(
        np.all(equal)
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected pre-export changes:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export values equal expected XPT "
    "after ONLY the three zero restorations."
)


# ============================================================
# 22. EXPORT CSV
# ============================================================

#
# NO:
#
# - round()
# - float_format
# - arbitrary recoding
# - missing-value filling
#
# Missing XPT values -> blank CSV.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 23. RE-READ CSV WITH ROUND-TRIP NUMERIC PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 24. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 25. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 92 columns numerically match "
    "expected XPT values exactly."
)


# ============================================================
# 26. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += (
        differences
    )


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing-position fidelity failure:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 27. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    raw_rows = list(
        csv.DictReader(f)
    )


if len(raw_rows) != 9666:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 28. STRICT RAW CSV TOKEN FIDELITY
# ============================================================

#
# Since MCQ_E has no genuine decimal variables:
#
# expected raw token =
#
# str(int(expected_source_value))
#
# Missing -> ""
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:

        print(example)


    raise ValueError(
        "MCQ_E raw CSV formatting differs."
    )


print(
    "PASS: Every raw CSV token matches "
    "the expected XPT-derived representation."
)


# ============================================================
# 29. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in EXPECTED_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Unwanted '.0' values:",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        dot_zero_examples
    )

    raise ValueError(
        "Unwanted .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 30. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 31. TINY ARTIFACT STRING CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: No tiny artifact remains."
)


# ============================================================
# 32. FINAL ZERO RECHECK
# ============================================================

for col, expected_count in (
    EXPECTED_ZERO_COUNTS.items()
):

    actual_count = int(
        csv_df[col]
        .eq(0)
        .sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"Final {col} zero count changed."
        )


final_total_zeros = int(
    csv_df[
        EXPECTED_COLUMNS
    ]
    .eq(0)
    .sum()
    .sum()
)


if final_total_zeros != 2884:

    raise ValueError(
        "Unexpected final zero count."
    )


print(
    "PASS: Final zero counts remain exact."
)


# ============================================================
# 33. FINAL KEY CDC RECHECK
# ============================================================

#
# MCQ150Q
#

s = csv_df["MCQ150Q"]

if not (
    int(s.between(0, 100).sum()) == 2364
    and
    int(s.eq(99999).sum()) == 3
    and
    int(s.isna().sum()) == 7299
):

    raise ValueError(
        "Final MCQ150Q distribution changed."
    )


#
# MCQ245B
#

s = csv_df["MCQ245B"]

if not (
    int(s.between(0, 365).sum()) == 3899
    and
    int(s.eq(99999).sum()) == 1
    and
    int(s.isna().sum()) == 5766
):

    raise ValueError(
        "Final MCQ245B distribution changed."
    )


#
# MCQ340
#

s = csv_df["MCQ340"]

if not (
    int(s.between(0, 20).sum()) == 1042
    and
    int(s.eq(999).sum()) == 47
    and
    int(s.isna().sum()) == 8577
):

    raise ValueError(
        "Final MCQ340 distribution changed."
    )


print(
    "PASS: Final CDC zero-capable fields unchanged."
)


# ============================================================
# 34. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("MCQ_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nValidated tiny -> zero:"
)

print(
    "MCQ150Q = 721"
)

print(
    "MCQ245B = 2,124"
)

print(
    "MCQ340  = 39"
)

print(
    "TOTAL   = 2,884"
)


print(
    "\nGenuine decimal variables:",
    0
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Unwanted .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "MCQ_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 MCQ_E XPT -> CSV
Rows: 9,666
Columns: 92
PASS: MCQ_E structure = 9,666 rows x 92 columns.
PASS: All 92 released variables are numeric.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifacts: 2,884

Tiny values by variable:
MCQ245B    2124
MCQ150Q     721
MCQ340       39

PASS: Exact MCQ_E tiny pattern confirmed.
      MCQ150Q = 721
      MCQ245B = 2,124
      MCQ340  = 39
      TOTAL   = 2,884
PASS: All tiny values occur only in CDC zero-permitted fields.

--- VALIDATED ZERO RESTORATION ---
MCQ150Q   : 721 tiny -> 0
MCQ245B   : 2,124 tiny -> 0
MCQ340    : 39 tiny -> 0
PASS: All 2,884 validated zeros restored.
PASS: Final zero counts are exact.
PASS: No unexpected zeros introduced.

Remaining tiny artifacts: 0
PASS: No tiny XPORT artifacts remain.

PASS: SEQN range = 41475-51623.
PASS: Missing SEQN = 0.
PASS: Duplicate SEQN = 0.

--- FRACTIONAL VALUE CHECK ---
Genuine fractional variables: {}
PASS: Genuine decimal variables = 0.

PASS: MCQ150Q
      0-100